# 使用 Microsoft Agent Framework 的人機互動流程

## 🎯 學習目標

在本筆記本中，您將學習如何使用 Microsoft Agent Framework 的 `RequestInfoExecutor` 實作 <strong>人機互動</strong> 流程。這個強大的模式讓您能暫停 AI 流程以收集人工輸入，使您的代理互動性更高，並讓人類掌控關鍵決策。

## 🔄 什麼是人機互動？

**人機互動 (Human-in-the-loop, HITL)** 是一種設計模式，AI 代理會暫停執行以請求人類輸入後才繼續。這對於以下情況至關重要：

- ✅ <strong>關鍵決策</strong> - 在進行重要行動前取得人工批准
- ✅ <strong>模糊狀況</strong> - 當 AI 不確定時讓人類澄清
- ✅ <strong>用戶偏好</strong> - 請用戶在多種選項中選擇
- ✅ <strong>合規與安全</strong> - 確保受監管操作具有人類監督
- ✅ <strong>互動體驗</strong> - 建立能回應用戶輸入的對話代理

## 🏗️ Microsoft Agent Framework 中的運作方式

框架提供 HITL 的三個關鍵元件：

1. **`RequestInfoExecutor`** - 一種特殊的執行器，會暫停流程並發出 `RequestInfoEvent`
2. **`RequestInfoMessage`** - 用於傳送給人類的類型化請求負載的基底類別
3. **`RequestResponse`** - 利用 `request_id` 對應人工回應與原始請求

**流程範式：**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 範例：具用戶確認的飯店預訂

我們將在條件流程基礎上加入，在建議替代目的地<strong>之前</strong>取得人工確認：

1. 使用者請求目的地（例如：「巴黎」）
2. `availability_agent` 檢查是否有空房
3. <strong>若無空房</strong> → `confirmation_agent` 詢問「您想看替代方案嗎？」
4. 使用 `RequestInfoExecutor` <strong>暫停</strong>流程
5. <strong>人工透過控制台輸入回應</strong>「是」或「否」
6. `decision_manager` 根據回應分路：
   - <strong>是</strong> → 顯示替代目的地
   - <strong>否</strong> → 取消預訂請求
7. 顯示最終結果

這展示了如何讓用戶掌控代理的建議！

---

讓我們開始吧！🚀


## 步驟 1：匯入所需的函式庫

我們匯入標準的 Agent 框架元件以及 <strong>人機互動專用類別</strong>：
- `RequestInfoExecutor` - 用於暫停工作流程以等待人工輸入的執行器
- `RequestInfoEvent` - 請求人工輸入時發出的事件
- `RequestInfoMessage` - 具型態化請求有效負載的基底類別
- `RequestResponse` - 將人工回覆與請求關聯起來
- `WorkflowOutputEvent` - 用於檢測工作流程輸出的事件


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 第 2 步：定義 Pydantic 模型以進行結構化輸出

這些模型定義了代理將返回的 <strong>結構</strong>。我們保留條件工作流程中的所有模型並新增：

**人類介入新項目：**
- `HumanFeedbackRequest` - `RequestInfoMessage` 的子類，定義發送給人類的請求負載
  - 包含 `prompt`（要詢問的問題）和 `destination`（關於該不可用城市的上下文）


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 步驟 3：建立飯店預訂工具

與條件工作流程中的工具相同 - 檢查目的地是否有可用房間。


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## 步驟 4：定義路由的條件函數

我們需要 <strong>四個條件函數</strong> 來實現我們的人類介入工作流程：

**來自條件工作流程：**
1. `has_availability_condition` - 酒店有空房時路由
2. `no_availability_condition` - 酒店沒有空房時路由

**人類介入的新條件：**
3. `user_wants_alternatives_condition` - 使用者對替代方案說「是」時路由
4. `user_declines_alternatives_condition` - 使用者對替代方案說「否」時路由


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 第五步：建立 Decision Manager Executor

這是 **human-in-the-loop 模式的核心**！`DecisionManager` 是一個自訂的 `Executor`，它：

1. **透過 `RequestResponse` 物件接收人類回饋**
2. <strong>處理使用者的決策</strong>（是/否）
3. <strong>透過傳送訊息給適當的代理人來引導工作流程</strong>

主要功能：
- 使用 `@handler` 裝飾器將方法公開為工作流程步驟
- 接收包含原始請求和使用者答案的 `RequestResponse[HumanFeedbackRequest, str]`
- 產生簡單的「是」或「否」訊息以觸發我們的條件函式


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## 第6步：建立自訂顯示執行器

與條件式工作流程相同的顯示執行器 - 產生最終結果作為工作流程輸出。


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## 第 7 步：載入環境變數

設定 LLM 用戶端（Microsoft Foundry / Azure OpenAI）。


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## 步驟 8：建立 AI 代理和執行者

我們創建了<strong>六個工作流程元件</strong>：

**代理（包裝在 AgentExecutor 中）：**
1. **availability_agent** - 使用工具檢查飯店可用性
2. **confirmation_agent** - 🆕 準備人工確認請求
3. **alternative_agent** - 建議替代城市（當使用者說「是」時）
4. **booking_agent** - 鼓勵訂房（當有房間可用時）
5. **cancellation_agent** - 🆕 處理取消訊息（當使用者說「否」時）

**特殊執行者：**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`，可暫停工作流程以等待人工輸入
7. **decision_manager** - 🆕 自訂執行者，根據人工回覆進行路由（已在上方定義）


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 第9步：建立帶有人類參與的工作流程

現在我們建構包含<strong>條件路由</strong>的人類參與路徑的工作流程圖：

**工作流程結構：**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**關鍵邊：**
- `availability_agent → confirmation_agent`（當沒有房間時）
- `confirmation_agent → prepare_human_request`（轉換類型）
- `prepare_human_request → request_info_executor`（暫停等待人類操作）
- `request_info_executor → decision_manager`（始終執行 - 提供 RequestResponse）
- `decision_manager → alternative_agent`（當使用者說「是」時）
- `decision_manager → cancellation_agent`（當使用者說「否」時）
- `availability_agent → booking_agent`（當房間可用時）
- 所有路徑皆終止於 `display_result`


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## 步驟 10：執行測試案例 1 - 無房態的城市（經人工確認的巴黎）

本測試展示了<strong>完整的人機回饋循環</strong>：

1. 請求巴黎的飯店
2. availability_agent 檢查 → 無房間
3. confirmation_agent 產生面向人的問題
4. request_info_executor <strong>暫停工作流程</strong>並發出 `RequestInfoEvent`
5. <strong>應用偵測事件並在主控台提示使用者</strong>
6. 使用者輸入「是」或「否」
7. 應用透過 `send_responses_streaming()` 傳送回應
8. decision_manager 根據回應決定路由
9. 顯示最終結果

**關鍵模式：**
- 第一次迭代使用 `workflow.run_stream()`
- 後續迭代使用 `workflow.send_responses_streaming(pending_responses)`
- 監聽 `RequestInfoEvent` 以偵測何時需要人工輸入
- 監聽 `WorkflowOutputEvent` 以捕捉最終結果


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## 步驟 11：執行測試案例 2 - 有庫存的城市（斯德哥爾摩 - 不需人工輸入）

此測試示範當有房間可用時的<strong>直接路徑</strong>：

1. 請求斯德哥爾摩的飯店
2. availability_agent 檢查 → 有房間✅
3. booking_agent 建議預訂
4. display_result 顯示確認訊息
5. **不需要人工輸入！**

當有房間可用時，工作流程會完全繞過人工介入路徑。


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## 主要收穫與人類介入最佳實踐

### ✅ 你學到了什麼：

#### 1. **RequestInfoExecutor 模式**
Microsoft Agent Framework 中的人類介入模式使用三個關鍵組件：
- `RequestInfoExecutor` - 暫停工作流程並發出事件
- `RequestInfoMessage` - 具類型化請求負載的基底類別（請繼承此類！）
- `RequestResponse` - 將人類回應與原始請求進行關聯

**關鍵理解：**
- `RequestInfoExecutor` 本身不會收集輸入 — 它只會暫停工作流程
- 你的應用程式代碼必須監聽 `RequestInfoEvent` 並收集輸入
- 你必須呼叫 `send_responses_streaming()`，傳入以 `request_id` 為鍵、使用者答案為值的字典

#### 2. <strong>串流執行模式</strong>
```python
# 第一次迭代
stream = workflow.run_stream(initial_request)

# 後續迭代（在人類輸入之後）
stream = workflow.send_responses_streaming(pending_responses)

# 一直處理事件
events = [event async for event in stream]
```

#### 3. <strong>事件驅動架構</strong>
監聽特定事件以控制工作流程：
- `RequestInfoEvent` - 需要人類輸入（工作流程暫停）
- `WorkflowOutputEvent` - 最終結果已完成（工作流程結束）
- `WorkflowStatusEvent` - 狀態變更（進行中、空閒且有待處理請求等）

#### 4. **使用 @handler 的自訂執行器**
`DecisionManager` 展示如何建立執行器：
- 使用 `@handler` 裝飾器將方法暴露為工作流程步驟
- 接收類型化訊息（例如 `RequestResponse[HumanFeedbackRequest, str]`）
- 透過發送訊息給其他執行器來路由工作流程
- 透過 `WorkflowContext` 存取上下文

#### 5. <strong>有條件的人為決策路由</strong>
你可以建立條件函式來評估人類回應：
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 真實世界應用：

1. <strong>核准工作流程</strong>
   - 處理費用報告前取得經理核准
   - 發送自動電子郵件前要求人類審查
   - 執行高價值交易前確認

2. <strong>內容審核</strong>
   - 標記有問題的內容供人類審查
   - 邀請審核人員對邊緣案例做出最終決策
   - 當 AI 信心低時升級至人工處理

3. <strong>客服服務</strong>
   - 讓 AI 自動處理日常問題
   - 複雜問題升級給人工客服
   - 詢問客戶是否想與人工客服對話

4. <strong>數據處理</strong>
   - 請人工解決資料歧義
   - 確認 AI 對不明文件的解釋
   - 讓使用者在多種有效解釋中做選擇

5. <strong>安全關鍵系統</strong>
   - 不可逆行動前要求人工確認
   - 存取敏感資料前取得批准
   - 法規產業（醫療、金融）中確認決策

6. <strong>互動代理人</strong>
   - 建立會持續提出追問的對話機器人
   - 創造引導使用者完成複雜程序的精靈
   - 設計與人類一步步合作的代理人

### 🔄 比較：有無人類介入

| 特性 | 有條件工作流程 | 人類介入工作流程 |
|---------|---------------------|---------------------------|
| <strong>執行</strong> | 單次 `workflow.run()` | 使用 `run_stream()` + `send_responses_streaming()` 的迴圈 |
| <strong>使用者輸入</strong> | 無（全自動） | 透過 `input()` 或 UI 互動提示 |
| <strong>組件</strong> | 代理 + 執行器 | + RequestInfoExecutor + DecisionManager |
| <strong>事件</strong> | 僅 AgentExecutorResponse | RequestInfoEvent、WorkflowOutputEvent 等 |
| <strong>暫停</strong> | 無暫停 | 工作流程在 RequestInfoExecutor 處暫停 |
| <strong>人工控制</strong> | 無人工控制 | 人類做出關鍵決策 |
| <strong>使用情境</strong> | 自動化 | 協作與監督 |

### 🚀 進階模式：

#### 多重人為決策點
同一工作流程內可有多個 `RequestInfoExecutor` 節點：
```python
.add_edge(agent1, request_info_1)  # 第一次人類決策
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # 第二次人類決策
.add_edge(decision_manager_2, final_agent)
```

#### 超時處理
為人類回答實作超時機制：
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # 預設為安全選項
```

#### 豐富的 UI 整合
避免使用 `input()`，整合 Web UI、Slack、Teams 等：
```python
if isinstance(event, RequestInfoEvent):
    # 發送通知到使用者偏好的頻道
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### 有條件的人類介入
僅在特定情況下請求人工輸入：
```python
def needs_human_approval_condition(message: Any) -> bool:
    # 僅當信心度低或數值高時，才轉介給人工處理
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ 最佳實踐：

1. **務必繼承 RequestInfoMessage**
   - 提供類型安全及驗證
   - 支援豐富上下文以利 UI 呈現
   - 清楚表達各請求類型意圖

2. <strong>使用具描述性的提示</strong>
   - 包含你所詢問事項的上下文
   - 解釋各選項的後果
   - 使問題簡單明確

3. <strong>處理意外輸入</strong>
   - 驗證使用者回應
   - 給予錯誤輸入的預設值
   - 提供清楚的錯誤訊息

4. **追蹤請求 ID**
   - 利用 request_id 與回應間的關聯
   - 不要嘗試手動管理狀態

5. <strong>設計非阻塞流程</strong>
   - 不要堵塞執行緒等待輸入
   - 全程使用非同步模式
   - 支援多併發工作流程實例

### 📚 相關概念：

- **Agent 中介軟體** - 攔截代理呼叫（上一筆筆記）
- <strong>工作流程狀態管理</strong> - 跨次執行保存狀態
- <strong>多代理協作</strong> - 將人類介入與代理團隊結合
- <strong>事件驅動架構</strong> - 建造以事件驅動的反應式系統

---

### 🎓 恭喜！

你已經精通 Microsoft Agent Framework 的人類介入工作流程！你現在已經能：
- ✅ 暫停工作流程以收集人類輸入
- ✅ 使用 RequestInfoExecutor 與 RequestInfoMessage
- ✅ 利用事件處理串流執行
- ✅ 建立擁有 @handler 的自訂執行器
- ✅ 根據人類決策路由工作流程
- ✅ 建造與人類協作的互動式 AI 代理

**這是構建值得信賴、可控 AI 系統的關鍵模式！** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
